In [0]:
# Databricks notebook source

from __future__ import annotations

import uuid

# ============================================================
# Widgets
# ============================================================
dbutils.widgets.text("catalog", "phc_txdot")

catalog = dbutils.widgets.get("catalog").strip()

# ============================================================
# Constants
# ============================================================
SCRIPT_NAME = "230_build_silver_dim_project.py"
RUN_ID = str(uuid.uuid4())

BASE_TABLE = f"{catalog}.silver.closed_bids_base_clean"
VENDOR_LATEST_TABLE = f"{catalog}.silver.closed_bids_vendor_item_latest"
ESTIMATE_CURRENT_TABLE = f"{catalog}.silver.closed_bids_estimate_item_current"
TARGET_TABLE = f"{catalog}.silver.dim_project"
RUN_LOG_TABLE = f"{catalog}.staging.pipeline_run_log"

# ============================================================
# Helpers
# ============================================================
def sql_escape(value: str) -> str:
    return value.replace("'", "''")


def log_run(status: str, row_count: int | None, message: str) -> None:
    row_count_sql = "NULL" if row_count is None else str(row_count)
    spark.sql(f"""
        INSERT INTO {RUN_LOG_TABLE}
        VALUES (
              '{RUN_ID}'
            , '{SCRIPT_NAME}'
            , '{sql_escape(status)}'
            , {row_count_sql}
            , '{sql_escape(message)}'
            , current_timestamp()
        )
    """)

# ============================================================
# Build dim_project
# ============================================================
try:
    spark.sql(f"""
    CREATE OR REPLACE TABLE {TARGET_TABLE}
    USING DELTA
    AS
    WITH project_source AS (
        SELECT *
        FROM {BASE_TABLE}
        WHERE project_id IS NOT NULL
    )

    , ranked AS (
        SELECT
              *
            , ROW_NUMBER() OVER (
                PARTITION BY project_id
                ORDER BY source_updated_at DESC NULLS LAST
            ) AS rn
        FROM project_source
    )

    , project_latest AS (
        SELECT *
        FROM ranked
        WHERE rn = 1
    )

    , vendor_stats AS (
        SELECT
              project_id
            , COUNT(DISTINCT vendor_name_standardized) AS vendor_count
            , COUNT(DISTINCT business_submission_key) AS submission_count
            , COUNT(DISTINCT CASE WHEN low_bidder_flag = TRUE THEN vendor_name_standardized END) AS low_bid_vendor_count
            , CASE
                  WHEN COUNT(DISTINCT CASE WHEN row_type = 'ENGINEER_ESTIMATE' THEN 1 END) > 0
                  THEN TRUE
                  ELSE FALSE
              END AS has_engineer_estimate
        FROM {BASE_TABLE}
        GROUP BY project_id
    )

    , vendor_latest_stats AS (
        SELECT
              project_id
            , COUNT(*) AS vendor_latest_item_count
            , COUNT(DISTINCT vendor_name_standardized) AS vendor_latest_vendor_count
        FROM {VENDOR_LATEST_TABLE}
        GROUP BY project_id
    )

    , estimate_stats AS (
        SELECT
              project_id
            , COUNT(*) AS estimate_current_item_count
            , MIN(engineer_project_total) AS min_engineer_project_total
            , MAX(engineer_project_total) AS max_engineer_project_total
        FROM {ESTIMATE_CURRENT_TABLE}
        GROUP BY project_id
    )

    SELECT
          md5(p.project_id) AS project_key
        , p.project_id
        , p.project_name
        , p.project_number
        , p.state_project_number
        , p.control_section_job_csj
        , p.controlling_project_id_ccsj
        , p.project_type
        , p.project_classification
        , p.project_actual_let_date
        , p.project_estimated_let_date
        , p.project_limits_from
        , p.project_limits_to
        , p.county
        , p.district_division
        , p.highway
        , p.short_description
        , p.federal_project_number

        , YEAR(p.project_actual_let_date) AS actual_let_year
        , MONTH(p.project_actual_let_date) AS actual_let_month

        , YEAR(p.project_estimated_let_date) AS estimated_let_year
        , MONTH(p.project_estimated_let_date) AS estimated_let_month

        , CASE
              WHEN p.project_actual_let_date IS NOT NULL
               AND p.project_estimated_let_date IS NOT NULL
              THEN DATEDIFF(p.project_actual_let_date, p.project_estimated_let_date)
              ELSE NULL
          END AS estimated_to_actual_let_day_diff

        , COALESCE(v.vendor_count, 0) AS vendor_count
        , COALESCE(v.submission_count, 0) AS submission_count
        , COALESCE(v.low_bid_vendor_count, 0) AS low_bid_vendor_count
        , COALESCE(v.has_engineer_estimate, FALSE) AS has_engineer_estimate

        , COALESCE(vl.vendor_latest_item_count, 0) AS vendor_latest_item_count
        , COALESCE(vl.vendor_latest_vendor_count, 0) AS vendor_latest_vendor_count

        , COALESCE(e.estimate_current_item_count, 0) AS estimate_current_item_count
        , e.min_engineer_project_total
        , e.max_engineer_project_total

        , CASE
              WHEN e.min_engineer_project_total IS NOT NULL
               AND e.max_engineer_project_total IS NOT NULL
               AND e.min_engineer_project_total <> e.max_engineer_project_total
              THEN TRUE
              ELSE FALSE
          END AS has_conflicting_engineer_project_totals

        , p.source_updated_at
        , p.ingested_at_utc

    FROM project_latest p
    LEFT JOIN vendor_stats v
        ON p.project_id = v.project_id
    LEFT JOIN vendor_latest_stats vl
        ON p.project_id = vl.project_id
    LEFT JOIN estimate_stats e
        ON p.project_id = e.project_id
    """)

    spark.sql(f"""
    COMMENT ON TABLE {TARGET_TABLE} IS
    'Project dimension built from silver.closed_bids_base_clean and enriched with vendor and engineer estimate statistics.'
    """)

    row_count = spark.sql(f"SELECT COUNT(*) AS row_count FROM {TARGET_TABLE}").collect()[0]["row_count"]

    log_run(
        "SUCCESS",
        row_count,
        f"Built {TARGET_TABLE} successfully."
    )

    print("=" * 70)
    print(f"Built {TARGET_TABLE}")
    print(f"Row count: {row_count:,}")
    print("=" * 70)

except Exception as exc:
    log_run("FAILED", None, str(exc))
    raise